In [1]:
import torch
import torch.nn as nn
from torch import tensor
import torch.nn.functional as F

In [18]:
attention = nn.MultiheadAttention(
    embed_dim=2,
    num_heads=2,
    batch_first=True,
    bias=False
)
attention.load_state_dict({
    "in_proj_weight": torch.tensor([
        # Q
        [1.0, 0.0],
        [0.0, 1.0],

        # K
        [2.0, 0.0],
        [0.0, 2.0],

        # V
        [3.0, 0.0],
        [0.0, 3.0],
    ]),
    "out_proj.weight": torch.tensor([
        [4.0, 0.0],
        [0.0, 4.0],
    ])
})

x = torch.tensor([
    [
        [0.2, 0.3],
    ]
])

output, weights = attention(
    query=x,
    key=x,
    value=x
)

output, weights

(tensor([[[2.4000, 3.6000]]], grad_fn=<TransposeBackward0>),
 tensor([[[1.]]], grad_fn=<MeanBackward1>))

In [24]:
x = torch.tensor([
    [
        [0.2, 0.3],
    ]
])

W_Q = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
])

W_K = torch.tensor([
    [2.0, 0.0],
    [0.0, 2.0],
])

W_V = torch.tensor([
    [3.0, 0.0],
    [0.0, 3.0],
])

W_O = torch.tensor([
    [4.0, 0.0],
    [0.0, 4.0],
])

Q = x @ W_Q.T
K = x @ W_K.T
V = x @ W_V.T

B, L, E = Q.shape
num_heads, head_dim = 2, 1

Qh = Q.view(B, L, num_heads, head_dim).transpose(1, 2) 
Kh = K.view(B, L, num_heads, head_dim).transpose(1, 2)
Vh = V.view(B, L, num_heads, head_dim).transpose(1, 2)

score = Qh @ Kh.transpose(-2, -1) / head_dim**0.5
weights = F.softmax(score, dim=-1)
attn = weights @ Vh

attn = attn.transpose(1, 2).contiguous().view(B, L, E)
output = attn @ W_O.T
output

tensor([[[2.4000, 3.6000]]])

In [29]:
Qh.shape, Q.shape

(torch.Size([1, 2, 1, 1]), torch.Size([1, 1, 2]))

In [28]:
weights.shape

torch.Size([1, 2, 1, 1])

### Check

In [31]:
attention = nn.MultiheadAttention(
    embed_dim=9,
    num_heads=2,
    batch_first=True,
    bias=False
)

AssertionError: embed_dim must be divisible by num_heads

In [32]:
attention = nn.MultiheadAttention(
    embed_dim=10,
    num_heads=2,
    batch_first=True,
    bias=False
)

### Check 2

In [33]:
attention = nn.MultiheadAttention(
    embed_dim=10,
    num_heads=2,
    batch_first=True,
    bias=False
)

x = torch.rand(
    1,  # batch
    2,  # sequence length
    10   # embed_dim
)

In [34]:
W_Q = attention.in_proj_weight[:10, :]
W_K = attention.in_proj_weight[10:20, :]
W_V = attention.in_proj_weight[20:30, :]

W_O = attention.out_proj.weight

In [37]:
Q = x @ W_Q.T
K = x @ W_K.T
V = x @ W_V.T

B, L, E = Q.shape
num_heads, head_dim = 2, 5

Qh = Q.view(B, L, num_heads, head_dim).transpose(1, 2) 
Kh = K.view(B, L, num_heads, head_dim).transpose(1, 2)
Vh = V.view(B, L, num_heads, head_dim).transpose(1, 2)

In [38]:
Qh.shape, Q.shape

(torch.Size([1, 2, 2, 5]), torch.Size([1, 2, 10]))